# Real data snippets: Spotify Sequential Skip Prediction Challenge

`./data_all/` is the public **Spotify Sequential Skip Prediction Challenge** dataset
(WSDM Cup 2019 / AIcrowd) -- 65 daily `log_0_YYYYMMDD_000000000000.csv` files, each
~500MB / ~3.4M rows. This notebook just lets you touch small pieces of it directly:
a raw row snippet, some quick session-level stats, and one full session pulled out
and run through the model pipeline built in `run_simulation.py`.

Two things this data does NOT have, worth remembering while poking around: no
identifier links sessions to the same listener (each `session_id` is independent),
and no per-play timestamp finer than `date` + `hour_of_day`.

The `data_all/subset/Clean_data_1/2/6.RData` files are corrupted/truncated (confirmed
separately) -- not used here.

## Parameters

In [1]:
import sys
sys.path.insert(0, "..")
import glob

import pandas as pd

from run_simulation import load_session_from_log, run_real_session

DATA_DIR = "../data_all"
LOG_FILES = sorted(glob.glob(f"{DATA_DIR}/log_0_*.csv"))
LOG_FILE = LOG_FILES[0]           # which day's file to peek at below
SNIPPET_ROWS = 2000               # small nrows read for the raw-row/stats snippets -- NOT the whole ~500MB file

print(f"{len(LOG_FILES)} log files found; using {LOG_FILE}")

60 log files found; using ../data_all/log_0_20180716_000000000000.csv


## Raw row snippet

Just the first `SNIPPET_ROWS` rows of one day's file, as-is.

In [2]:
snippet = pd.read_csv(LOG_FILE, nrows=SNIPPET_ROWS)
snippet.head(10)

,session_id,session_position,session_length,track_id_clean,skip_1,skip_2,skip_3,not_skipped,context_switch,no_pause_before_play,...,long_pause_before_play,hist_user_behavior_n_seekfwd,hist_user_behavior_n_seekback,hist_user_behavior_is_shuffle,hour_of_day,date,premium,context_type,hist_user_behavior_reason_start,hist_user_behavior_reason_end
0,1_00001238-3f7a-4fd2-b52e-af4237a41ed9,1,20,t_91eb6576-0f1f-4394-92d8-560d7b8f632b,True,True,True,False,0,0,...,0,0,0,True,5,2018-07-16,True,editorial_playlist,fwdbtn,fwdbtn
1,1_00001238-3f7a-4fd2-b52e-af4237a41ed9,2,20,t_338cb0f3-0035-418f-ab2d-3f9342815aaa,True,True,True,False,0,1,...,0,0,0,True,5,2018-07-16,True,editorial_playlist,fwdbtn,fwdbtn
2,1_00001238-3f7a-4fd2-b52e-af4237a41ed9,3,20,t_98e73634-8f33-4275-b544-3451553f431e,False,True,True,False,0,1,...,0,0,0,True,5,2018-07-16,True,editorial_playlist,fwdbtn,endplay
3,1_00001238-3f7a-4fd2-b52e-af4237a41ed9,4,20,t_2fd6279a-06cd-4d5d-9e54-1912f51bb3c3,False,False,False,True,0,1,...,0,0,0,False,6,2018-07-16,True,editorial_playlist,clickrow,trackdone
4,1_00001238-3f7a-4fd2-b52e-af4237a41ed9,5,20,t_de860a61-bb57-4386-9d5a-6ba111c4c266,False,False,False,True,0,1,...,0,0,0,False,6,2018-07-16,True,editorial_playlist,trackdone,trackdone
5,1_00001238-3f7a-4fd2-b52e-af4237a41ed9,6,20,t_0e3dec82-10b4-49f1-8c2e-cd19249f7d2c,False,False,False,True,0,1,...,0,0,0,False,6,2018-07-16,True,editorial_playlist,trackdone,trackdone
6,1_00001238-3f7a-4fd2-b52e-af4237a41ed9,7,20,t_eb0063cc-948f-41d4-9dab-bf4b0e70d21e,False,False,False,True,0,1,...,0,0,0,False,6,2018-07-16,True,editorial_playlist,trackdone,trackdone
7,1_00001238-3f7a-4fd2-b52e-af4237a41ed9,8,20,t_4319026b-d2b5-478d-bf37-8f2873645c5c,False,False,False,True,0,1,...,0,0,0,False,6,2018-07-16,True,editorial_playlist,trackdone,trackdone
8,1_00001238-3f7a-4fd2-b52e-af4237a41ed9,9,20,t_e8fa3463-7ef7-4ca5-8e65-2df979012c34,False,False,True,False,0,1,...,0,0,0,False,6,2018-07-16,True,editorial_playlist,trackdone,trackdone
9,1_00001238-3f7a-4fd2-b52e-af4237a41ed9,10,20,t_9a03f300-6504-4c7a-92b1-4cd88354ee74,False,False,True,False,0,1,...,0,0,0,False,6,2018-07-16,True,editorial_playlist,trackdone,fwdbtn


In [3]:
snippet.dtypes

session_id                         object
session_position                    int64
session_length                      int64
track_id_clean                     object
skip_1                               bool
skip_2                               bool
skip_3                               bool
not_skipped                          bool
context_switch                      int64
no_pause_before_play                int64
short_pause_before_play             int64
long_pause_before_play              int64
hist_user_behavior_n_seekfwd        int64
hist_user_behavior_n_seekback       int64
hist_user_behavior_is_shuffle        bool
hour_of_day                         int64
date                               object
premium                              bool
context_type                       object
hist_user_behavior_reason_start    object
hist_user_behavior_reason_end      object
dtype: object

## Quick stats on the snippet

Session lengths, skip rates, and how many sessions in this snippet already have an
in-session repeated track (the thing Algorithm A/A2/A3/A4 need to produce any
`aesthetic_basis`/`evaluation` value at all).

In [4]:
print("unique sessions in snippet:", snippet["session_id"].nunique())
print()
print("session_length distribution:")
print(snippet.drop_duplicates("session_id")["session_length"].describe())
print()
print("skip-column rates:")
print(snippet[["skip_1", "skip_2", "skip_3", "not_skipped"]].mean())

unique sessions in snippet: 120

session_length distribution:
count    120.000000
mean      16.691667
std        3.763593
min       10.000000
25%       13.000000
50%       19.000000
75%       20.000000
max       20.000000
Name: session_length, dtype: float64

skip-column rates:
skip_1         0.3995
skip_2         0.5080
skip_3         0.6415
not_skipped    0.3375
dtype: float64


In [5]:
has_repeat = snippet.groupby("session_id")["track_id_clean"].agg(lambda s: s.duplicated().any())
print(f"sessions (fully captured in this snippet) with an in-session repeat: "
      f"{has_repeat.sum()} / {len(has_repeat)}")

sessions (fully captured in this snippet) with an in-session repeat: 71 / 120


## One full session, pulled out and run through the pipeline

Picks the first session (fully contained in the snippet above) that has a repeat
track, loads its complete row set via `load_session_from_log` (streams just that
session out of the full file), and shows both the raw session rows and what
`run_real_session` (Algorithm A4 / Design 5's algorithm) produces from it.

In [6]:
candidate_sessions = has_repeat[has_repeat].index
session_id = candidate_sessions[0]
print("session_id:", session_id)

session_df = load_session_from_log(LOG_FILE, session_id)
session_df[["session_position", "song_id", "skip_1", "skip_2", "skip_3", "not_skipped", "real_skip", "hour_timestamp"]]

session_id: 1_00001238-3f7a-4fd2-b52e-af4237a41ed9


,session_position,song_id,skip_1,skip_2,skip_3,not_skipped,real_skip,hour_timestamp
0,1,t_91eb6576-0f1f-4394-92d8-560d7b8f632b,True,True,True,False,1.0,2018-07-16 05:00:00
1,2,t_338cb0f3-0035-418f-ab2d-3f9342815aaa,True,True,True,False,1.0,2018-07-16 05:00:00
2,3,t_98e73634-8f33-4275-b544-3451553f431e,False,True,True,False,1.0,2018-07-16 05:00:00
3,4,t_2fd6279a-06cd-4d5d-9e54-1912f51bb3c3,False,False,False,True,0.0,2018-07-16 06:00:00
4,5,t_de860a61-bb57-4386-9d5a-6ba111c4c266,False,False,False,True,0.0,2018-07-16 06:00:00
5,6,t_0e3dec82-10b4-49f1-8c2e-cd19249f7d2c,False,False,False,True,0.0,2018-07-16 06:00:00
6,7,t_eb0063cc-948f-41d4-9dab-bf4b0e70d21e,False,False,False,True,0.0,2018-07-16 06:00:00
7,8,t_4319026b-d2b5-478d-bf37-8f2873645c5c,False,False,False,True,0.0,2018-07-16 06:00:00
8,9,t_e8fa3463-7ef7-4ca5-8e65-2df979012c34,False,False,True,False,0.0,2018-07-16 06:00:00
9,10,t_9a03f300-6504-4c7a-92b1-4cd88354ee74,False,False,True,False,0.0,2018-07-16 06:00:00


In [7]:
result = run_real_session(session_df, time_mode="tick",
                           config_path="../config.yaml",
                           output_path="../results/data/real_session_snippet_demo.csv")
result[["trial", "session_position", "actual_song_id", "actual_activation",
        "aesthetic_basis", "evaluation", "skip_1", "skip_2", "skip_3", "not_skipped", "real_skip"]]

,trial,session_position,actual_song_id,actual_activation,aesthetic_basis,evaluation,skip_1,skip_2,skip_3,not_skipped,real_skip
0,17,18,t_338cb0f3-0035-418f-ab2d-3f9342815aaa,-2.036302,2.637776,0.0,False,False,False,True,0.0
